# 01 · Template photometry and colour diagrams

**What this does.** Builds one row per DIA object from the delivered Fink LSST Light Static alerts: a template PSF
magnitude in every band the object alerted in, plus its Fink labels and observing programme. It then draws the three
colour diagrams.

| Figure | Content |
|---|---|
| `fig1_overview_all_dia` | r vs g−i, g−r vs u−g, r−i vs g−r for all colour-eligible DIA objects |
| `fig2_by_fink_cats_class` | the same, one row per Fink CATS label |
| `fig3_by_fink_simbad_crossmatch` | the same, one row per SIMBAD group (Fink crossmatch within 1″) |

**Method in one paragraph.** Every alert is one detection in one band and carries `templateFlux`: forced PSF photometry
on that band's difference-imaging template, centred at the DIA object position (nJy). For each object and band we take
the median `templateFlux` and `templateFluxErr` over its alerts. We keep the band if the flux is positive and the median
S/N ≥ 5, and convert it to AB with m = 31.4 − 2.5 log10(f/nJy). Fink labels come from the object's latest alert. The
SIMBAD type is the latest successful crossmatch; "crossmatch failed" means every alert's crossmatch errored.

**Inputs:** the analytical catalog (`CATALOG` in `_setup.py`), opened read-only, and its pinned raw Parquet files.
**Outputs:** `RUN_ROOT/01_template_colours/` holding `figures/`, `tables/` and `summary.json`. The per-object table
`tables/dia_object_template_photometry.parquet` is the input to notebooks 02–05.

Run time on Arnor: about 3 minutes.

In [ ]:
# Shared configuration: paths, run folder, helpers (edit notebooks/science/_setup.py, not this cell).
from _setup import *
describe_config()
import alert_colour_diagrams as acd
out = step_dir("01_template_colours")

## Step 1 · Read the catalog and build the per-object table
Opens the catalog read-only (raw-file fingerprints are re-verified), aggregates template fluxes per object and band,
attaches the latest Fink labels, the SIMBAD group and the observing programme (from `target_name`).

In [ ]:
inputs = acd.load_catalog_inputs(CATALOG, DATA_ROOT, temp_dir=TEMP_DIR)
df = inputs["df"]
print("cohort (requested UTC dates):", inputs["window"])
print("source rows by population   :", inputs["pop"])
print("raw Parquet files           :", len(inputs["raw_files"]))
print("DIA objects                 :", f"{len(df):,}")
print("usable template bands       :", {b: int(df[f'ok_{b}'].sum()) for b in acd.BANDS})
print("panels a, c (g, r, i)       :", int(df.panel_a.sum()), "   panel b (u, g, r):", int(df.panel_b.sum()))
inputs["band_pop"]

## Step 2 · How stable is the template flux across an object's alerts?
(max − min) / median of `templateFlux` per band, for objects with at least two alerts in that band.

In [ ]:
stab = acd.template_stability(df)
stab

## Step 3 · Count tables
Written to `tables/`: the per-object table, counts by Fink label and by SIMBAD group, the SIMBAD type → group map,
template-flux summaries per band, and the Fink software versions present in the data.

In [ ]:
cats, simb = acd.write_tables(inputs, out / "tables")
display(cats)
display(simb)

## Step 4 · Figures 1–3

In [ ]:
fig_paths = acd.make_figures(df, out / "figures", inputs["window"])
show(fig_paths)

## Step 5 · Provenance summary
Written to `summary.json`: code version, catalog SHA256, package versions, cuts and headline counts.

In [ ]:
summary = acd.build_summary(inputs, stab, cats, simb, out.name, code_version(), CATALOG, round(time.time() - T0, 1))
save_json(summary, out / "summary.json")
{k: summary[k] for k in ("dia_objects", "panel_a_and_c_gri_objects", "panel_b_ugr_objects", "panel_b_by_field")}

**Reference values for the 2026-02-25 → 2026-07-13 cohort** (useful when re-running on the same catalog):
3,487,175 DIA objects; 8,196 in the g, r, i panels; 1,050 in the u, g, r panel, all in deep-drilling fields.